In [1]:
%reload_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import pandas as pd
from functools import reduce
from tqdm.auto import tqdm
from pandarallel import pandarallel
pandarallel.initialize()

from rdkit import Chem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit.Chem.Draw import rdDepictor
rdDepictor.SetPreferCoordGen(True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

from paper_style import export_consistent_mol_figures

clear_output()

In [2]:
smis = {
    'enterobactin': 'C1[C@@H](C(=O)OC[C@@H](C(=O)OC[C@@H](C(=O)O1)NC(=O)C2=C(C(=CC=C2)O)O)NC(=O)C3=C(C(=CC=C3)O)O)NC(=O)C4=C(C(=CC=C4)O)O',
    'salmochelin_s4': 'C1[C@@H](C(=O)OC[C@@H](C(=O)OC[C@@H](C(=O)O1)NC(=O)C2=C(C(=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)O)O)NC(=O)C4=C(C(=CC(=C4)[C@H]5[C@@H]([C@H]([C@@H]([C@H](O5)CO)O)O)O)O)O)NC(=O)C6=C(C(=CC=C6)O)O',
    'cyclic_trichrysobactin': 'C1[C@@H](C(=O)OC[C@@H](C(=O)OC[C@@H](C(=O)O1)NC(=O)[C@@H](CCCCN)NC(=O)C2=C(C(=CC=C2)O)O)NC(=O)[C@@H](CCCCN)NC(=O)C3=C(C(=CC=C3)O)O)NC(=O)[C@@H](CCCCN)NC(=O)C4=C(C(=CC=C4)O)O',
    'mirubactin': 'NC(=NCCC[C@@H](NC(=O)C1=C(O)C(O)=CC=C1)C(=O)N[C@H](CCCN(O)C=O)C(=O)O)NC(=O)C1=C(O)C(O)=CC=C1'}

mol_dict = {k:Chem.MolFromSmiles(v) for k, v in smis.items()}
# export_consistent_mol_figures(mol_dict, prefix='fig1', canvas_w=400)

In [ ]:
# Get maximum common substructure
from rdkit.Chem import rdFMCS
res = rdFMCS.FindMCS(list(mol_dict.values())[:3])
mcs_smarts = res.smartsString
mcs_mol = Chem.MolFromSmarts(res.smartsString)

export_consistent_mol_figures({'mcs': mcs_mol}, fixed_font_size=14, canvas_w=250)

In [5]:
import sys
sys.path.append("/home/mattia/geass")
from new_geass import Geass
geass = Geass()

frags = []
for idx, mol in enumerate(mol_dict.values(), start=1):
    suffix = chr(ord('a') + idx - 1)
    frags.extend(geass.get_fragments(mol))
    # geass.show_connection_graph()
    # geass.export_connection_graph(f'figures/fig1{suffix}-graph.svg')

# Keep one fragment per MUID
unique_frags = {}
for frag in frags:
    if frag.HasProp('MUID'):
        key = frag.GetProp('MUID')
    unique_frags[key] = frag

export_consistent_mol_figures(unique_frags, prefix='frag', 
    fixed_font_size=6, canvas_w=50)

Saved figures/frag_C00065.svg
Saved figures/frag_C00196.svg
Saved figures/frag_C00221.svg
Saved figures/frag_C00739.svg
Saved figures/frag_C20850.svg
Saved figures/frag_C00792.svg


In [ ]:
sys.path.append('/home/mattia/geass')

cluster_graphs = []
for mol in list(mol_dict.values())[:3]:
    geass.get_fragments(mol)
    g = geass.create_connection_graph()
    cluster_graphs.append(g)

import variational_graph as vg
V = vg.build_variational_graph(cluster_graphs)
vg.show_variational_graph(V)